# Tutorial: Colab Qwen2.5-Coder Text-to-SQL Smoke Test

Audience:
- You already have the CLI project and want a Colab notebook that runs the same 20-row smoke-test pipeline end to end.

Prerequisites:
- Google Colab with GPU runtime enabled.
- Either `/content/Model-FineTuning.zip`, an extracted `/content/Model-FineTuning` folder, or a Git URL for cloning the repo.
- A Hugging Face token configured in Colab if model or dataset access requires authentication.

Learning goals:
- Inspect the Gretel Text-to-SQL dataset schema.
- Build a cleaned 20-row Spider-aligned smoke dataset.
- Fine-tune `Qwen/Qwen2.5-Coder-7B-Instruct` with Unsloth QLoRA.
- Run a post-training inference smoke test and save the adapter artifacts.


## Outline

1. Bootstrap the repo inside Colab
2. Install Unsloth and project dependencies
3. Inspect the dataset and generate the 20-row cleaned split
4. Train the QLoRA adapter
5. Run a smoke-test inference query
6. Zip the adapter for download or Drive upload


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path('/content/Model-FineTuning')
ZIP_PATH = Path('/content/Model-FineTuning.zip')
REPO_URL = ''  # Optional. Set this if the repo is not already present in /content.
HF_TOKEN = ''  # Optional. Set this if you need authenticated HF access.

def run(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print(f'\n$ {cmd}')
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        executable='/bin/bash',
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {cmd}')
    return result

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

if PROJECT_DIR.exists():
    print(f'Using existing project at {PROJECT_DIR}')
elif ZIP_PATH.exists():
    print(f'Found uploaded zip at {ZIP_PATH}. Extracting to /content ...')
    run(f'unzip -o {ZIP_PATH} -d /content')
    nested_dir = PROJECT_DIR / 'Model-FineTuning'
    if nested_dir.exists() and not (PROJECT_DIR / 'scripts').exists():
        print('Detected nested Model-FineTuning directory. Flattening it ...')
        for item in nested_dir.iterdir():
            target = PROJECT_DIR / item.name
            if target.exists():
                continue
            shutil.move(str(item), str(target))
        shutil.rmtree(nested_dir)
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(f'Zip extracted, but {PROJECT_DIR} was not created.')
elif REPO_URL:
    run(f'git clone {REPO_URL} {PROJECT_DIR}')
else:
    raise FileNotFoundError(
        'Project repo not found. Upload /content/Model-FineTuning.zip, extract /content/Model-FineTuning, or set REPO_URL.'
    )

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())


## Step 1 - Verify the Colab runtime

This confirms that you are on a GPU runtime before spending time on installs and training. A T4 is enough for the 20-row smoke test.


In [ ]:
run('nvidia-smi', check=False)


## Step 2 - Install dependencies

This installs the repo dependencies, then refreshes Unsloth with the current Colab-friendly upstream install path. If your runtime disconnects, rerun this cell first.


In [ ]:
run('python -m pip install -U pip setuptools wheel')
run('python -m pip install -r requirements/colab.txt')
run('python -m pip uninstall -y unsloth unsloth_zoo', check=False)
run("python -m pip install --upgrade --no-cache-dir 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'")
run("python -m pip install --upgrade --no-cache-dir 'git+https://github.com/unslothai/unsloth-zoo.git'")
run('python -m pip install -e .')


## Step 3 - Inspect the Gretel dataset schema

This is the guardrail against dataset drift. The preprocessing script checks that the expected columns still exist before building the smoke split.


In [ ]:
run('python scripts/preprocess_gretel.py --inspect-only')


## Step 4 - Build the 20-row cleaned smoke dataset

The pipeline keeps read-only SQLite-compatible tasks, strips inserts from the schema context, deduplicates examples, and writes `train.jsonl` and `val.jsonl` under `data/processed/colab_smoke`.


In [ ]:
run('python scripts/preprocess_gretel.py --config configs/colab_smoke.yaml')


In [ ]:
summary_path = Path('data/processed/colab_smoke/summary.json')
train_path = Path('data/processed/colab_smoke/train.jsonl')
summary = json.loads(summary_path.read_text())
first_train = json.loads(train_path.read_text().splitlines()[0])
summary


In [ ]:
{
    'question': first_train['sql_prompt'],
    'schema_ddl': first_train['schema_ddl'],
    'target_sql': first_train['sql'],
}


## Step 5 - Train the smoke-test adapter

This run is intentionally small and slightly overfit. Its job is to verify prompt formatting, assistant-only masking, checkpointing, adapter saving, and post-training inference.


In [ ]:
run('python scripts/train_unsloth.py --config configs/colab_smoke.yaml')


## Step 6 - Smoke-test inference

The notebook uses a seen training example first. That is deliberate for smoke testing: if this cell fails, debug the pipeline before trying unseen prompts.


In [ ]:
schema_file = Path('/content/colab_smoke_schema.sql')
schema_file.write_text(first_train['schema_ddl'])
cmd = [
    'python', 'scripts/infer_unsloth.py',
    '--config', 'configs/colab_smoke.yaml',
    '--model-path', 'artifacts/colab_smoke_qwen25_coder/adapter',
    '--question', first_train['sql_prompt'],
    '--schema-file', str(schema_file),
    '--strategy', 'advanced_reasoning',
]
result = subprocess.run(cmd, text=True, capture_output=True)
print('Question:', first_train['sql_prompt'])
print('Expected:', first_train['sql'])
print('Predicted:', result.stdout.strip())
if result.stderr.strip():
    print('stderr:', result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Inference failed with code {result.returncode}')


## Step 7 - Save artifacts

This zips the LoRA adapter so you can download it from Colab or copy it to Drive. The default smoke preset does not merge the model, because the goal here is just end-to-end verification.


In [ ]:
adapter_dir = Path('artifacts/colab_smoke_qwen25_coder/adapter')
zip_path = Path('/content/colab_smoke_qwen25_coder_adapter.zip')
if adapter_dir.exists():
    run(f'cd {adapter_dir.parent} && zip -r {zip_path} adapter')
else:
    raise FileNotFoundError(f'Missing adapter directory: {adapter_dir}')
print('Zipped adapter:', zip_path)


## Pitfalls and extensions

Common failure points:
- If model loading says the Unsloth build does not support the model, rerun the install cell.
- If you hit OOM on a T4, lower `model.max_seq_length` to `1024` or keep batch size at `1`.
- If the model emits explanations instead of SQL, keep `include_explanation: false` and `temperature=0.0`.

Next steps after the smoke run:
- Switch to `configs/vast_5k.yaml` on Vast.ai for the real 5k run.
- Keep the same training format: schema + question -> SQL only.
- Benchmark the adapter or merged model on Spider 1.0 using your existing `advanced_reasoning` inference strategy.


## Exercise

Create a temporary Colab config for a 100-row experiment with fewer epochs. The next cell writes the override file so you can test a slightly less toy setting without touching the checked-in presets.


In [ ]:
import yaml

base_cfg = yaml.safe_load(Path('configs/colab_smoke.yaml').read_text())
base_cfg['dataset']['sample_size'] = 100
base_cfg['dataset']['output_root'] = 'data/processed/colab_100'
base_cfg['training']['output_dir'] = 'artifacts/colab_100_qwen25_coder'
base_cfg['training']['num_train_epochs'] = 3
override_path = Path('configs/colab_100.yaml')
override_path.write_text(yaml.safe_dump(base_cfg, sort_keys=False))
print('Wrote', override_path)
print('Run next: python scripts/preprocess_gretel.py --config configs/colab_100.yaml')
print('Then:     python scripts/train_unsloth.py --config configs/colab_100.yaml')
